In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 02 - Prétraitement des Données Cliniques et Génomiques\n",
        "\n",
        "Ce notebook prétraite les données cliniques et génomiques pour la modélisation XAI.\n",
        "\n",
        "## Objectifs\n",
        "1. Nettoyer les données cliniques (imputation, encodage).\n",
        "2. Normaliser les données génomiques (log2, quantile normalization).\n",
        "3. Sélectionner les features pertinentes.\n",
        "4. Fusionner les données cliniques et génomiques.\n",
        "5. Sauvegarder les données prétraitées pour la modélisation."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# --- Imports ---\n",
        "import yaml\n",
        "import pandas as pd\n",
        "import numpy as np\n",
        "import os\n",
        "from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer\n",
        "from sklearn.preprocessing import RobustScaler, StandardScaler, QuantileTransformer\n",
        "from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif\n",
        "from sklearn.compose import ColumnTransformer\n",
        "from sklearn.pipeline import Pipeline\n",
        "from sklearn.model_selection import train_test_split\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "\n",
        "# Configuration des visualisations\n",
        "%matplotlib inline\n",
        "plt.style.use('seaborn-v0_8')\n",
        "sns.set_palette(\"husl\")\n",
        "sns.set_context(\"notebook\", font_scale=1.1)\n",
        "\n",
        "# Charger la configuration\n",
        "with open(\"../config/config.yaml\", \"r\") as f:\n",
        "    config = yaml.safe_load(f)\n",
        "\n",
        "# Chemins et paramètres\n",
        "data_dir = config[\"data\"][\"data_dir\"]\n",
        "pancan_dir = config[\"data\"].get(\"pancan_dir\")\n",
        "target_column = config[\"data\"][\"target_column\"]\n",
        "scaling = config[\"preprocessing\"][\"scaling\"]\n",
        "genomic_scaling = config[\"preprocessing\"][\"genomic_scaling\"]\n",
        "n_genes = config[\"data\"].get(\"n_genes\", 1000)\n",
        "random_state = config[\"project\"][\"random_state\"]"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Chargement des Données Brutes"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Charger les données cliniques\n",
        "clinical_path = os.path.join(data_dir, config[\"data\"][\"clinical_file\"])\n",
        "df_clinical = pd.read_csv(clinical_path, sep=\"\\t\")\n",
        "\n",
        "print(f\"Données cliniques chargées: {df_clinical.shape}\")\n",
        "display(df_clinical.head())"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Prétraitement des Données Cliniques"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Séparer X et y\n",
        "X_clinical = df_clinical.drop(columns=[target_column])\n",
        "y = df_clinical[target_column]\n",
        "\n",
        "# Identifier les colonnes numériques et catégorielles\n",
        "numeric_cols = X_clinical.select_dtypes(include=[np.number]).columns\n",
        "categorical_cols = X_clinical.select_dtypes(include=['object', 'category']).columns\n",
        "\n",
        "print(f\"Variables numériques: {len(numeric_cols)}\")\n",
        "print(f\"Variables catégorielles: {len(categorical_cols)}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Imputation des valeurs manquantes (numériques)\n",
        "if len(numeric_cols) > 0:\n",
        "    imputer = IterativeImputer(max_iter=10, random_state=random_state)\n",
        "    X_clinical[numeric_cols] = imputer.fit_transform(X_clinical[numeric_cols])\n",
        "\n",
        "# Imputation des valeurs manquantes (catégorielles)\n",
        "if len(categorical_cols) > 0:\n",
        "    imputer = SimpleImputer(strategy=\"most_frequent\")\n",
        "    X_clinical[categorical_cols] = imputer.fit_transform(X_clinical[categorical_cols])\n",
        "\n",
        "# Scaling des variables numériques\n",
        "if len(numeric_cols) > 0:\n",
        "    if scaling == \"robust\":\n",
        "        scaler = RobustScaler()\n",
        "    else:\n",
        "        scaler = StandardScaler()\n",
        "    X_clinical[numeric_cols] = scaler.fit_transform(X_clinical[numeric_cols])\n",
        "\n",
        "# Encodage des variables catégorielles (One-Hot Encoding)\n",
        "if len(categorical_cols) > 0:\n",
        "    X_clinical = pd.get_dummies(X_clinical, columns=categorical_cols)\n",
        "\n",
        "print(f\"Données cliniques après prétraitement: {X_clinical.shape}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. Prétraitement des Données Génomiques (Expression)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Charger les données d'expression BRCA\n",
        "expression_path = os.path.join(data_dir, config[\"data\"][\"genomic_files\"][2])  # data_mrna_seq_v2_rsem.tsv\n",
        "df_expression = pd.read_csv(expression_path, sep=\"\\t\", index_col=0)\n",
        "\n",
        "print(f\"Données d'expression BRCA: {df_expression.shape}\")\n",
        "\n",
        "# Transposer pour avoir les échantillons en lignes et les gènes en colonnes\n",
        "df_expression = df_expression.T\n",
        "print(f\"Après transposition: {df_expression.shape}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Imputation des valeurs manquantes (KNN pour les données génomiques)\n",
        "imputer = KNNImputer(n_neighbors=5)\n",
        "df_expression_imputed = pd.DataFrame(\n",
        "    imputer.fit_transform(df_expression),\n",
        "    index=df_expression.index,\n",
        "    columns=df_expression.columns\n",
        ")\n",
        "\n",
        "# Normalisation\n",
        "if genomic_scaling == \"log2\":\n",
        "    # Log2(FPKM+1) pour les données d'expression\n",
        "    df_expression_normalized = np.log2(df_expression_imputed + 1)\n",
        "    print(\"Normalisation log2 appliquée\")\n",
        "elif genomic_scaling == \"quantile\":\n",
        "    # Normalisation par quantiles\n",
        "    scaler = QuantileTransformer(output_distribution=\"normal\")\n",
        "    df_expression_normalized = pd.DataFrame(\n",
        "        scaler.fit_transform(df_expression_imputed),\n",
        "        index=df_expression_imputed.index,\n",
        "        columns=df_expression_imputed.columns\n",
        "    )\n",
        "    print(\"Normalisation par quantiles appliquée\")\n",
        "else:\n",
        "    # StandardScaler par défaut\n",
        "    scaler = StandardScaler()\n",
        "    df_expression_normalized = pd.DataFrame(\n",
        "        scaler.fit_transform(df_expression_imputed),\n",
        "        index=df_expression_imputed.index,\n",
        "        columns=df_expression_imputed.columns\n",
        "    )\n",
        "    print(\"Normalisation standard appliquée\")\n",
        "\n",
        "print(f\"Données d'expression après normalisation: {df_expression_normalized.shape}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4. Sélection des Gènes Pertinents"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Sélection des gènes les plus variables\n",
        "gene_variances = df_expression_normalized.var(axis=0)\n",
        "top_genes = gene_variances.nlargest(n_genes).index\n",
        "df_expression_selected = df_expression_normalized[top_genes]\n",
        "\n",
        "print(f\"Gènes sélectionnés: {len(top_genes)} (sur {df_expression_normalized.shape[1]})\")\n",
        "print(\"Top 10 gènes par variance:\")\n",
        "display(gene_variances.nlargest(10))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5. Fusion des Données Cliniques et Génomiques"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Vérifier que les échantillons correspondent entre données cliniques et génomiques\n",
        "common_samples = set(X_clinical.index) & set(df_expression_selected.index)\n",
        "if len(common_samples) == 0:\n",
        "    # Si les index ne correspondent pas, utiliser les noms de colonnes comme échantillons\n",
        "    common_samples = set(X_clinical.index) & set(df_expression_selected.columns)\n",
        "    if len(common_samples) == 0:\n",
        "        raise ValueError(\"Aucun échantillon commun trouvé entre les données cliniques et génomiques.\")\n",
        "    else:\n",
        "        # Transposer df_expression_selected pour aligner les échantillons\n",
        "        df_expression_selected = df_expression_selected.T\n",
        "        common_samples = set(X_clinical.index) & set(df_expression_selected.index)\n",
        "\n",
        "# Filtrer pour les échantillons communs\n",
        "X_clinical_filtered = X_clinical.loc[list(common_samples)]\n",
        "df_expression_filtered = df_expression_selected.loc[list(common_samples)]\n",
        "\n",
        "print(f\"Échantillons communs: {len(common_samples)}\")\n",
        "print(f\"Données cliniques filtrées: {X_clinical_filtered.shape}\")\n",
        "print(f\"Données génomiques filtrées: {df_expression_filtered.shape}\")"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Fusionner les données cliniques et génomiques\n",
        "X = pd.concat([X_clinical_filtered, df_expression_filtered], axis=1)\n",
        "\n",
        "# Filtrer y pour les échantillons communs\n",
        "y_filtered = y.loc[list(common_samples)]\n",
        "\n",
        "print(f\"Données fusionnées: {X.shape}\")\n",
        "print(f\"Variable cible filtrée: {y_filtered.shape}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6. Séparation en Jeux d'Entraînement/Test"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Séparer en train/test\n",
        "X_train, X_test, y_train, y_test = train_test_split(\n",
        "    X, y_filtered,\n",
        "    test_size=config[\"data\"][\"test_size\"],\n",
        "    random_state=random_state,\n",
        "    stratify=y_filtered\n",
        ")\n",
        "\n",
        "print(f\"Train set: {X_train.shape}\")\n",
        "print(f\"Test set: {X_test.shape}\")\n",
        "print(f\"Proportion de la cible dans le train: {y_train.mean():.2f}\")\n",
        "print(f\"Proportion de la cible dans le test: {y_test.mean():.2f}\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 7. Sauvegarde des Données Prétraitées"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Créer le dossier de sortie\n",
        "os.makedirs(\"../data/processed\", exist_ok=True)\n",
        "\n",
        "# Sauvegarder les données prétraitées\n",
        "X_train.to_parquet(\"../data/processed/X_train.parquet\")\n",
        "X_test.to_parquet(\"../data/processed/X_test.parquet\")\n",
        "y_train.to_frame().to_parquet(\"../data/processed/y_train.parquet\")\n",
        "y_test.to_frame().to_parquet(\"../data/processed/y_test.parquet\")\n",
        "\n",
        "# Sauvegarder la liste des gènes sélectionnés\n",
        "with open(\"../data/processed/selected_genes.txt\", \"w\") as f:\n",
        "    f.write(\"\\n\".join(top_genes))\n",
        "\n",
        "# Sauvegarder un résumé du prétraitement\n",
        "with open(\"../reports/preprocessing/summary.txt\", \"w\") as f:\n",
        "    f.write(\"=== RÉSUMÉ DU PRÉTRAITEMENT ===\\n\\n\")\n",
        "    f.write(f\"Données cliniques initiales: {df_clinical.shape}\\n\")\n",
        "    f.write(f\"Données cliniques après prétraitement: {X_clinical.shape}\\n\")\n",
        "    f.write(f\"Données d'expression initiales: {df_expression.shape}\\n\")\n",
        "    f.write(f\"Données d'expression après prétraitement: {df_expression_normalized.shape}\\n\")\n",
        "    f.write(f\"Gènes sélectionnés: {len(top_genes)}\\n\")\n",
        "    f.write(f\"Données fusionnées: {X.shape}\\n\")\n",
        "    f.write(f\"Train set: {X_train.shape}\\n\")\n",
        "    f.write(f\"Test set: {X_test.shape}\\n\")\n",
        "\n",
        "print(\"✅ Prétraitement terminé. Données sauvegardées dans data/processed/\")"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.9.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# 01 - Exploration des Données BRCA_TCGA et PANCAN\n',
    '\n',
    "Ce notebook explore les données cliniques et génomiques pour la prédiction d'issues cliniques avec XAI.\n",
    '\n',
    '## Objectifs\n',
    '1. Charger les données cliniques et génomiques.\n',
    '2. Explorer les distributions et relations entre variables.\n',
    '3. Identifier les problèmes potentiels (valeurs manquantes, déséquilibres).\n',
    '4. Générer des visualisations pour le rapport.']},
  {'cell_type': 'code',
   'execution_count': None,
   'metadata': {},
   'outputs': [],
   'source': ['# --- Imports ---\n',
    'import yaml\n',
    'import pandas as pd\n',
    'import numpy as np\n',
    'import os\n',
    'import matplotlib.pyplot as plt\n',
    'import seaborn as sns\n',
    'from IPython.display import display\n',
    'from pathlib import Path\n',
    '\n',
    '# Configuration des visualisations\n',
    '%matplotlib inline\